# Clean & Build Dataset — Leak-Fixed Version

**What changed from `clean_and_build_dataset.ipynb`:**
- All features are cutoff-date-aware (only data available at prediction time is used)
- 5 mitigation techniques applied: `FILTER`, `DISTINCT ON`, expanding window, `created_date <= cutoff`, and drop
- 8 challenge features dropped (junction tables lack date columns)
- `position_id_encoded` excluded (fold-safe target encoding only)
- Two output CSVs: `dataset_at_creation_clean.csv` and `dataset_at_halfway_clean.csv`

**Two cutoffs per task:**
- `creation_cutoff = created_date` (prediction at task creation)
- `halfway_cutoff = GREATEST(created_date, start_date + duration/2)` (prediction at task midpoint)

In [1]:
import os, warnings
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
import pandas as pd
import numpy as np
from datetime import datetime

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 250)

load_dotenv()
engine = create_engine(os.getenv('DB_URL'))
FIXED_CUTOFF = pd.Timestamp('2026-07-14').normalize()

STATUS_ORDER = {'not_started': 0, 'ongoing': 1, 'in_progress': 1, 'completed': 2, 'terminated': 3, 'archived': 4}
APR_ORDER = {'pending': 0, 'in_review': 1, 'approved': 2, 'rejected': 3}
MA_STATUS_ORDER = {'not_started': 0, 'ongoing': 1, 'completed': 2, 'terminated': 3}
KPI_STATUS_ORDER = {'not_started': 0, 'ongoing': 1, 'completed': 2, 'terminated': 3, 'archived': 4}

def q(sql):
    return pd.read_sql(sql, engine)

def tz_naive(ser):
    ser = pd.to_datetime(ser, errors='coerce')
    if hasattr(ser.dt, 'tz') and ser.dt.tz is not None:
        ser = ser.dt.tz_localize(None)
    return ser

print('Ready.')

Ready.


---
## 1. Base Tasks with Target & Cutoffs

Each task gets two cutoff timestamps:
- `creation_cutoff = created_date`
- `halfway_cutoff = GREATEST(created_date, start_date + (end_date - start_date) / 2)`

The `GREATEST` cap handles retroactive scheduling (76% of tasks have `start_date < created_date`).

In [2]:
base = q("""
    SELECT t.id, t.status, t.approval_status, t.lead_approval_status,
           t.weight_level, t.is_planned::int AS is_planned, t.risk_mapping,
           t.start_date, t.end_date, t.actual_end_date,
           t.created_date, t.updated_date,
           t.major_activity_id, t.created_by_id, t.position_id, t.department_id,
           CASE WHEN t.derived_from_cross_department_assignment_id IS NOT NULL THEN 1 ELSE 0 END AS is_cross_dept,
           t.created_date AS creation_cutoff,
           GREATEST(t.created_date, t.start_date + (t.end_date - t.start_date) / 2) AS halfway_cutoff
    FROM tasks_task t
    WHERE t.start_date IS NOT NULL AND t.end_date IS NOT NULL
""")
for col in ['start_date', 'end_date', 'actual_end_date', 'created_date', 'updated_date', 'creation_cutoff', 'halfway_cutoff']:
    base[col] = tz_naive(base[col])
print(f'Base tasks: {len(base)}')

Base tasks: 13895


In [3]:
# Three-tier target: actual_end_date > first_completed_at > updated_date
first_completed = q("""
    SELECT history_relation_id AS task_id, MIN(history_date) AS first_completed_at
    FROM tasks_task_history WHERE status = 'completed'
    GROUP BY history_relation_id
""")
first_completed['first_completed_at'] = tz_naive(first_completed['first_completed_at'])
base = base.merge(first_completed, left_on='id', right_on='task_id', how='left')
base.drop(columns=['task_id'], inplace=True)
print(f'Tasks with completion history: {first_completed["task_id"].nunique()}')

Tasks with completion history: 5327


In [4]:
completion_date = base['actual_end_date'].copy()
completion_date = completion_date.fillna(base['first_completed_at'].dt.normalize())
completion_date = completion_date.fillna(base['updated_date'].dt.normalize())

conditions = [
    (base['status'] == 'completed') & (base['actual_end_date'].notna()),
    (base['status'] == 'completed') & (base['actual_end_date'].isna()) & (base['first_completed_at'].notna()),
    (base['status'] == 'completed') & (base['actual_end_date'].isna()) & (base['first_completed_at'].isna()),
    ~base['status'].isin(['completed', 'terminated', 'archived']) & (base['end_date'] < FIXED_CUTOFF),
]
source_labels = ['actual_end_date', 'history_completion', 'updated_date', 'open_task']
base['target_source'] = np.select(conditions, source_labels, default='status_based')

overdue_cond = [
    (base['status'] == 'completed') & (completion_date > base['end_date']),
    ~base['status'].isin(['completed', 'terminated', 'archived']) & (base['end_date'] < FIXED_CUTOFF),
]
base['calculated_overdue'] = np.select(overdue_cond, [1, 1], default=0)

print(f'Overdue rate: {base["calculated_overdue"].mean():.2%}')
print('Target source:')
print(base['target_source'].value_counts())

Overdue rate: 47.81%
Target source:
target_source
actual_end_date       7444
updated_date          5427
status_based           722
history_completion     231
open_task               71
Name: count, dtype: int64


---
## 2. Derived Features (Static — Same at Both Timepoints)

These features depend only on `created_date`, `start_date`, `end_date` — they are fixed at creation and don't change.

In [5]:
base['planned_duration'] = (base['end_date'] - base['start_date']).dt.days
base['creation_to_planned_start'] = (base['start_date'] - base['created_date']).dt.days
base['created_dow'] = base['created_date'].dt.dayofweek
base['created_is_weekend'] = (base['created_dow'] >= 5).astype(int)
base['created_is_friday'] = (base['created_dow'] == 4).astype(int)
base['created_month'] = base['created_date'].dt.month
base['created_quarter'] = base['created_date'].dt.quarter
base['days_since_update_halfway'] = (FIXED_CUTOFF - base['updated_date']).dt.days
print('Derived features computed.')

Derived features computed.


---
## 3. Revisions — FILTER (history_date <= cutoff)

**Leak:** Original query counts ALL history rows. Revisions made after the prediction point leak future info.

**Fix:** `COUNT(*) FILTER (WHERE history_date <= creation_cutoff)` and `<= halfway_cutoff`. At creation, no history exists yet → count = 0 for all tasks.

In [6]:
rev = q("""
    SELECT th.history_relation_id AS task_id,
           COUNT(*) FILTER (WHERE th.history_date <= b.creation_cutoff) AS num_revisions_at_creation,
           COUNT(*) FILTER (WHERE th.history_date <= b.halfway_cutoff) AS num_revisions_at_halfway,
           MAX(th.history_date) FILTER (WHERE th.history_date <= b.creation_cutoff) AS last_revision_at_creation,
           MAX(th.history_date) FILTER (WHERE th.history_date <= b.halfway_cutoff) AS last_revision_at_halfway
    FROM tasks_task_history th
    JOIN (SELECT id, created_date AS creation_cutoff,
                 GREATEST(created_date, start_date + (end_date - start_date) / 2) AS halfway_cutoff
          FROM tasks_task WHERE start_date IS NOT NULL AND end_date IS NOT NULL) b
      ON b.id = th.history_relation_id
    GROUP BY th.history_relation_id
""")
for col in ['last_revision_at_creation', 'last_revision_at_halfway']:
    if col in rev.columns and rev[col].notna().any():
        rev[col] = tz_naive(rev[col])

created_by_id = base.set_index('id')['created_date']
rev['revision_frequency_at_creation'] = 0.0
rev['revision_recency_at_creation'] = rev['task_id'].map(
    lambda tid: (FIXED_CUTOFF - created_by_id.loc[tid]).days if tid in created_by_id.index else 0
)
rev['revision_frequency_at_halfway'] = rev['num_revisions_at_halfway'].fillna(0).astype(float)
rev['revision_recency_at_halfway'] = (
    (FIXED_CUTOFF - tz_naive(rev['last_revision_at_halfway'])).dt.days
    if rev['last_revision_at_halfway'].notna().any() else 0
).fillna(0).astype(int)

base = base.merge(
    rev[['task_id', 'num_revisions_at_creation', 'num_revisions_at_halfway',
         'revision_frequency_at_creation', 'revision_frequency_at_halfway',
         'revision_recency_at_creation', 'revision_recency_at_halfway']],
    left_on='id', right_on='task_id', how='left'
)
for col in ['num_revisions_at_creation', 'num_revisions_at_halfway']:
    base[col] = base[col].fillna(0).astype(int)
for col in ['revision_frequency_at_creation', 'revision_frequency_at_halfway',
            'revision_recency_at_creation', 'revision_recency_at_halfway']:
    base[col] = base[col].fillna(0)
base.drop(columns=['task_id'], inplace=True)

print(f'Tasks with revisions at creation: {(base["num_revisions_at_creation"] > 0).sum()} (expect 0)')
print(f'Tasks with revisions at halfway:  {(base["num_revisions_at_halfway"] > 0).sum()}')

Tasks with revisions at creation: 0 (expect 0)
Tasks with revisions at halfway:  3250


---
## 4. Subtasks — FILTER (created_date <= cutoff)

**Leak:** Original counts ALL subtasks, including ones created after the prediction point.

**Fix:** `FILTER (WHERE created_date <= cutoff)`. Subtask status at cutoff is used for completion/overdue rates.

In [7]:
sub = q("""
    SELECT st.task_id,
           COUNT(*) FILTER (WHERE st.created_date <= b.creation_cutoff) AS num_subtasks_at_creation,
           COUNT(*) FILTER (WHERE st.created_date <= b.halfway_cutoff) AS num_subtasks_at_halfway,
           SUM(CASE WHEN st.status = 'completed' AND st.created_date <= b.creation_cutoff THEN 1 ELSE 0 END) AS num_completed_at_creation,
           SUM(CASE WHEN st.status = 'completed' AND st.created_date <= b.halfway_cutoff THEN 1 ELSE 0 END) AS num_completed_at_halfway,
           SUM(CASE WHEN st.is_overdue = TRUE AND st.created_date <= b.creation_cutoff THEN 1 ELSE 0 END) AS num_overdue_at_creation,
           SUM(CASE WHEN st.is_overdue = TRUE AND st.created_date <= b.halfway_cutoff THEN 1 ELSE 0 END) AS num_overdue_at_halfway
    FROM tasks_sub_task st
    JOIN (SELECT id, created_date AS creation_cutoff,
                 GREATEST(created_date, start_date + (end_date - start_date) / 2) AS halfway_cutoff
          FROM tasks_task WHERE start_date IS NOT NULL AND end_date IS NOT NULL) b
      ON b.id = st.task_id
    GROUP BY st.task_id
""")
sub['subtask_completion_pct_at_creation'] = (sub['num_completed_at_creation'] / sub['num_subtasks_at_creation'].replace(0, np.nan)).fillna(0)
sub['subtask_completion_pct_at_halfway'] = (sub['num_completed_at_halfway'] / sub['num_subtasks_at_halfway'].replace(0, np.nan)).fillna(0)
sub['subtask_overdue_rate_at_creation'] = (sub['num_overdue_at_creation'] / sub['num_subtasks_at_creation'].replace(0, np.nan)).fillna(0)
sub['subtask_overdue_rate_at_halfway'] = (sub['num_overdue_at_halfway'] / sub['num_subtasks_at_halfway'].replace(0, np.nan)).fillna(0)

base = base.merge(
    sub[['task_id', 'num_subtasks_at_creation', 'num_subtasks_at_halfway',
         'subtask_completion_pct_at_creation', 'subtask_completion_pct_at_halfway',
         'subtask_overdue_rate_at_creation', 'subtask_overdue_rate_at_halfway']],
    left_on='id', right_on='task_id', how='left'
)
base['has_subtasks_at_creation'] = (base['num_subtasks_at_creation'] > 0).astype(int)
base['has_subtasks_at_halfway'] = (base['num_subtasks_at_halfway'] > 0).astype(int)
for col in ['num_subtasks_at_creation', 'num_subtasks_at_halfway']:
    base[col] = base[col].fillna(0).astype(int)
for col in ['subtask_completion_pct_at_creation', 'subtask_completion_pct_at_halfway',
            'subtask_overdue_rate_at_creation', 'subtask_overdue_rate_at_halfway']:
    base[col] = base[col].fillna(0.0)
base.drop(columns=['task_id'], inplace=True)

print(f'Tasks with subtasks at creation: {(base["num_subtasks_at_creation"] > 0).sum()}')
print(f'Tasks with more subtasks at halfway: {(base["num_subtasks_at_halfway"] > base["num_subtasks_at_creation"]).sum()}')

Tasks with subtasks at creation: 0
Tasks with more subtasks at halfway: 91


---
## 5. Status Lookup — DISTINCT ON (history_date <= cutoff)

**Leak:** Original uses CURRENT task status, which may have changed after the prediction point.

**Fix:** `DISTINCT ON (task_id) ... WHERE history_date <= cutoff ORDER BY history_date DESC` — gets the latest status that was recorded AT or before the cutoff.

In [8]:
st_c = q("""
    SELECT DISTINCT ON (th.history_relation_id)
        th.history_relation_id AS task_id,
        th.status AS status_at_creation,
        th.approval_status AS approval_status_at_creation,
        th.lead_approval_status AS lead_approval_status_at_creation
    FROM tasks_task_history th
    JOIN (SELECT id, created_date AS creation_cutoff
          FROM tasks_task WHERE start_date IS NOT NULL AND end_date IS NOT NULL) b
      ON b.id = th.history_relation_id
    WHERE th.history_date <= b.creation_cutoff
    ORDER BY th.history_relation_id, th.history_date DESC
""")
st_h = q("""
    SELECT DISTINCT ON (th.history_relation_id)
        th.history_relation_id AS task_id,
        th.status AS status_at_halfway,
        th.approval_status AS approval_status_at_halfway,
        th.lead_approval_status AS lead_approval_status_at_halfway
    FROM tasks_task_history th
    JOIN (SELECT id, GREATEST(created_date, start_date + (end_date - start_date) / 2) AS halfway_cutoff
          FROM tasks_task WHERE start_date IS NOT NULL AND end_date IS NOT NULL) b
      ON b.id = th.history_relation_id
    WHERE th.history_date <= b.halfway_cutoff
    ORDER BY th.history_relation_id, th.history_date DESC
""")

base['s_creation'] = base['id'].map(st_c.set_index('task_id')['status_at_creation']).fillna(base['status'])
base['s_halfway'] = base['id'].map(st_h.set_index('task_id')['status_at_halfway']).fillna(base['status'])
base['a_creation'] = base['id'].map(st_c.set_index('task_id')['approval_status_at_creation']).fillna(base['approval_status'])
base['a_halfway'] = base['id'].map(st_h.set_index('task_id')['approval_status_at_halfway']).fillna(base['approval_status'])
base['l_creation'] = base['id'].map(st_c.set_index('task_id')['lead_approval_status_at_creation']).fillna(base['lead_approval_status'])
base['l_halfway'] = base['id'].map(st_h.set_index('task_id')['lead_approval_status_at_halfway']).fillna(base['lead_approval_status'])

base['status_encoded_creation'] = base['s_creation'].map(STATUS_ORDER).fillna(1).astype(int)
base['status_encoded_halfway'] = base['s_halfway'].map(STATUS_ORDER).fillna(1).astype(int)
base['approval_status_encoded_creation'] = base['a_creation'].map(APR_ORDER).fillna(1).astype(int)
base['approval_status_encoded_halfway'] = base['a_halfway'].map(APR_ORDER).fillna(1).astype(int)
base['lead_approval_status_encoded_creation'] = base['l_creation'].map(APR_ORDER).fillna(1).astype(int)
base['lead_approval_status_encoded_halfway'] = base['l_halfway'].map(APR_ORDER).fillna(1).astype(int)
base.drop(columns=['s_creation', 's_halfway', 'a_creation', 'a_halfway', 'l_creation', 'l_halfway'], inplace=True)

not_started_c = (base['status_encoded_creation'] == 0).sum()
not_started_h = (base['status_encoded_halfway'] == 0).sum()
print(f'Status=not_started at creation: {not_started_c}/{len(base)}')
print(f'Status=not_started at halfway:  {not_started_h}/{len(base)}')

Status=not_started at creation: 133/13895
Status=not_started at halfway:  2649/13895


---
## 6. Major Activity & KPI Features

MA info (status) is static — an MA's status doesn't change per-task. MA/KPI revisions use `FILTER (history_date <= cutoff)`. KPI status uses `DISTINCT ON` history lookup.

In [9]:
ma = q("""
    SELECT ma.id AS major_activity_id, ma.status AS ma_status,
           ma.approval_status AS ma_approval_status, ma.kpi_id
    FROM tasks_major_activity ma
""")
base = base.merge(ma, on='major_activity_id', how='left')
base['ma_status_encoded_creation'] = base['ma_status'].map(MA_STATUS_ORDER).fillna(1).astype(int)
base['ma_status_encoded_halfway'] = base['ma_status'].map(MA_STATUS_ORDER).fillna(1).astype(int)
base['ma_approval_status_encoded_creation'] = base['ma_approval_status'].map(APR_ORDER).fillna(1).astype(int)
base['ma_approval_status_encoded_halfway'] = base['ma_approval_status'].map(APR_ORDER).fillna(1).astype(int)
print('MA info loaded.')

MA info loaded.


In [10]:
mar = q("""
    SELECT b.id,
           COUNT(*) FILTER (WHERE mah.history_date <= b.creation_cutoff) AS num_ma_revisions_at_creation,
           COUNT(*) FILTER (WHERE mah.history_date <= b.halfway_cutoff) AS num_ma_revisions_at_halfway
    FROM (SELECT id, major_activity_id, created_date AS creation_cutoff,
                 GREATEST(created_date, start_date + (end_date - start_date) / 2) AS halfway_cutoff
          FROM tasks_task WHERE start_date IS NOT NULL AND end_date IS NOT NULL) b
    LEFT JOIN tasks_major_activity_history mah ON mah.id = b.major_activity_id
    GROUP BY b.id
""")
base = base.merge(mar[['id', 'num_ma_revisions_at_creation', 'num_ma_revisions_at_halfway']], on='id', how='left')
base['num_ma_revisions_at_creation'] = base['num_ma_revisions_at_creation'].fillna(0).astype(int)
base['num_ma_revisions_at_halfway'] = base['num_ma_revisions_at_halfway'].fillna(0).astype(int)

print(f'MA revisions: creation mean={base["num_ma_revisions_at_creation"].mean():.2f}, '
      f'halfway mean={base["num_ma_revisions_at_halfway"].mean():.2f}')

MA revisions: creation mean=2.49, halfway mean=3.23


In [11]:
# KPI status via DISTINCT ON history lookup at each cutoff
kp = q("""
    SELECT b.id,
           (SELECT kh.is_overdue::int FROM tasks_kpi_history kh
            WHERE kh.id = ma.kpi_id AND kh.history_date <= b.creation_cutoff
            ORDER BY kh.history_date DESC LIMIT 1) AS kpi_is_overdue_creation,
           (SELECT kh.status FROM tasks_kpi_history kh
            WHERE kh.id = ma.kpi_id AND kh.history_date <= b.creation_cutoff
            ORDER BY kh.history_date DESC LIMIT 1) AS kpi_status_creation,
           (SELECT kh.is_overdue::int FROM tasks_kpi_history kh
            WHERE kh.id = ma.kpi_id AND kh.history_date <= b.halfway_cutoff
            ORDER BY kh.history_date DESC LIMIT 1) AS kpi_is_overdue_halfway,
           (SELECT kh.status FROM tasks_kpi_history kh
            WHERE kh.id = ma.kpi_id AND kh.history_date <= b.halfway_cutoff
            ORDER BY kh.history_date DESC LIMIT 1) AS kpi_status_halfway
    FROM (SELECT t.id, t.major_activity_id, t.created_date AS creation_cutoff,
                 GREATEST(t.created_date, t.start_date + (t.end_date - t.start_date) / 2) AS halfway_cutoff
          FROM tasks_task t WHERE t.start_date IS NOT NULL AND t.end_date IS NOT NULL) b
    LEFT JOIN tasks_major_activity ma ON ma.id = b.major_activity_id
""")
base['kpi_is_overdue_flag_creation'] = kp['kpi_is_overdue_creation'].fillna(0).astype(int)
base['kpi_is_overdue_flag_halfway'] = kp['kpi_is_overdue_halfway'].fillna(0).astype(int)
base['kpi_status_ordinal_creation'] = kp['kpi_status_creation'].fillna('ongoing').map(KPI_STATUS_ORDER).fillna(1).astype(int)
base['kpi_status_ordinal_halfway'] = kp['kpi_status_halfway'].fillna('ongoing').map(KPI_STATUS_ORDER).fillna(1).astype(int)

# KPI revisions with FILTER
kpi_rev = q("""
    SELECT b.id,
           COALESCE((SELECT COUNT(*) FROM tasks_kpi_history kh
                     WHERE kh.id = ma.kpi_id AND kh.history_date <= b.creation_cutoff), 0)::int AS num_kpi_revisions_at_creation,
           COALESCE((SELECT COUNT(*) FROM tasks_kpi_history kh
                     WHERE kh.id = ma.kpi_id AND kh.history_date <= b.halfway_cutoff), 0)::int AS num_kpi_revisions_at_halfway
    FROM (SELECT t.id, t.major_activity_id, t.created_date AS creation_cutoff,
                 GREATEST(t.created_date, t.start_date + (t.end_date - t.start_date) / 2) AS halfway_cutoff
          FROM tasks_task t WHERE t.start_date IS NOT NULL AND t.end_date IS NOT NULL) b
    LEFT JOIN tasks_major_activity ma ON ma.id = b.major_activity_id
""")
base['num_kpi_revisions_at_creation'] = kpi_rev['num_kpi_revisions_at_creation'].fillna(0).astype(int)
base['num_kpi_revisions_at_halfway'] = kpi_rev['num_kpi_revisions_at_halfway'].fillna(0).astype(int)

print(f'KPI features loaded. Overdue at creation: {base["kpi_is_overdue_flag_creation"].sum()}, '
      f'at halfway: {base["kpi_is_overdue_flag_halfway"].sum()}')

KPI features loaded. Overdue at creation: 36, at halfway: 38


---
## 7. Comments — FILTER (created_date <= cutoff)

**Leak:** Original counts ALL comments on a task, including those made after the prediction point.

**Fix:** `COUNT(*) FILTER (WHERE created_date <= creation_cutoff / halfway_cutoff)`.

In [12]:
tc = q("""
    SELECT cc.object_id AS task_id,
           COUNT(*) FILTER (WHERE cc.created_date <= b.creation_cutoff) AS task_comment_count_at_creation,
           COUNT(*) FILTER (WHERE cc.created_date <= b.halfway_cutoff) AS task_comment_count_at_halfway
    FROM comments_comment cc
    JOIN (SELECT id, created_date AS creation_cutoff,
                 GREATEST(created_date, start_date + (end_date - start_date) / 2) AS halfway_cutoff
          FROM tasks_task WHERE start_date IS NOT NULL AND end_date IS NOT NULL) b
      ON b.id = cc.object_id
    WHERE cc.content_type_id = 24
    GROUP BY cc.object_id
""")
base = base.merge(tc, left_on='id', right_on='task_id', how='left')
base['task_comment_count_at_creation'] = base['task_comment_count_at_creation'].fillna(0).astype(int)
base['task_comment_count_at_halfway'] = base['task_comment_count_at_halfway'].fillna(0).astype(int)
base.drop(columns=['task_id'], inplace=True)

print(f'Tasks with comments at creation: {(base["task_comment_count_at_creation"] > 0).sum()}')
print(f'Tasks with comments at halfway:  {(base["task_comment_count_at_halfway"] > 0).sum()}')

Tasks with comments at creation: 0
Tasks with comments at halfway:  3


---
## 8. Sub-Task Status Churn — FILTER (history_date <= cutoff)

**Leak:** Original counts ALL sub-task status changes, including those after the prediction point.

**Fix:** `COUNT(DISTINCT status) FILTER (WHERE history_date <= cutoff)`.

In [13]:
sch = q("""
    SELECT st.task_id,
           AVG(sh.num_status_changes_at_creation) AS avg_sub_status_changes_at_creation,
           AVG(sh.num_status_changes_at_halfway) AS avg_sub_status_changes_at_halfway
    FROM (
        SELECT sth.id AS sub_task_id,
               COUNT(DISTINCT sth.status) FILTER (WHERE sth.history_date <= b.creation_cutoff) AS num_status_changes_at_creation,
               COUNT(DISTINCT sth.status) FILTER (WHERE sth.history_date <= b.halfway_cutoff) AS num_status_changes_at_halfway
        FROM tasks_sub_task_history sth
        JOIN tasks_sub_task st ON st.id = sth.id
        JOIN (SELECT id, created_date AS creation_cutoff,
                     GREATEST(created_date, start_date + (end_date - start_date) / 2) AS halfway_cutoff
              FROM tasks_task WHERE start_date IS NOT NULL AND end_date IS NOT NULL) b
          ON b.id = st.task_id
        GROUP BY sth.id
    ) sh
    JOIN tasks_sub_task st ON st.id = sh.sub_task_id
    GROUP BY st.task_id
""")
base = base.merge(sch, left_on='id', right_on='task_id', how='left')
base['avg_sub_status_changes_at_creation'] = base['avg_sub_status_changes_at_creation'].fillna(0)
base['avg_sub_status_changes_at_halfway'] = base['avg_sub_status_changes_at_halfway'].fillna(0)
base.drop(columns=['task_id'], inplace=True)

print(f'avg_sub_status_changes: creation mean={base["avg_sub_status_changes_at_creation"].mean():.3f}, '
      f'halfway mean={base["avg_sub_status_changes_at_halfway"].mean():.3f}')

avg_sub_status_changes: creation mean=0.000, halfway mean=0.006


---
## 9. Cross-Department Pair Flag

Static feature — determined at creation from `derived_from_cross_department_assignment_id`.

In [14]:
cd = q("""
    SELECT t.id AS task_id, 1 AS cross_dept_pair_exists
    FROM tasks_task t
    JOIN tasks_cross_department_assignments cda ON cda.id = t.derived_from_cross_department_assignment_id
""")
base = base.merge(cd, left_on='id', right_on='task_id', how='left')
base['cross_dept_pair_exists'] = base['cross_dept_pair_exists'].fillna(0).astype(int)
base.drop(columns=['task_id'], inplace=True)
print(f'Cross-dept pairs: {base["cross_dept_pair_exists"].sum()}')

Cross-dept pairs: 188


---
## 10. Group Aggregates — Expanding Window (Self-Excluding)

**Leak (temporal + self):** Original includes ALL department tasks + the current task itself.

**Fix:** `AVG(is_overdue) OVER (PARTITION BY dept_id ORDER BY created_date ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING)` — computes the running average of ALL PREVIOUS tasks, excluding the current row.

In [15]:
dept_agg = q("""
    SELECT id, dept_past_overdue_rate, dept_avg_revisions FROM (
        SELECT t.id,
               AVG(CASE WHEN t.status = 'completed' AND t.actual_end_date IS NOT NULL AND t.actual_end_date > t.end_date THEN 1
                        WHEN t.status = 'completed' AND t.actual_end_date IS NULL AND t.updated_date::date > t.end_date THEN 1
                        WHEN t.status NOT IN ('completed', 'terminated', 'archived') AND t.end_date < '2026-07-14'::date THEN 1
                        ELSE 0 END)
                   OVER (PARTITION BY COALESCE(p.department_id, t.department_id)
                         ORDER BY t.created_date ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS dept_past_overdue_rate,
               AVG(COALESCE(rev_cnt.num_revisions, 0))
                   OVER (PARTITION BY COALESCE(p.department_id, t.department_id)
                         ORDER BY t.created_date ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS dept_avg_revisions
        FROM tasks_task t
        LEFT JOIN basedata_position p ON p.id = t.position_id
        LEFT JOIN (SELECT history_relation_id AS task_id, COUNT(*) AS num_revisions
                   FROM tasks_task_history WHERE history_relation_id IS NOT NULL GROUP BY history_relation_id) rev_cnt
          ON rev_cnt.task_id = t.id
        WHERE t.start_date IS NOT NULL AND t.end_date IS NOT NULL
    ) sub
""")
emp_agg = q("""
    SELECT id, emp_past_overdue_rate FROM (
        SELECT t.id,
               AVG(CASE WHEN t.status = 'completed' AND t.actual_end_date IS NOT NULL AND t.actual_end_date > t.end_date THEN 1
                        WHEN t.status = 'completed' AND t.actual_end_date IS NULL AND t.updated_date::date > t.end_date THEN 1
                        WHEN t.status NOT IN ('completed', 'terminated', 'archived') AND t.end_date < '2026-07-14'::date THEN 1
                        ELSE 0 END)
                   OVER (PARTITION BY p.user_id
                         ORDER BY t.created_date ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS emp_past_overdue_rate
        FROM tasks_task t
        LEFT JOIN basedata_position p ON p.id = t.position_id
        WHERE t.start_date IS NOT NULL AND t.end_date IS NOT NULL
    ) sub
""")
pos_agg = q("""
    SELECT id, pos_past_overdue_rate FROM (
        SELECT t.id,
               AVG(CASE WHEN t.status = 'completed' AND t.actual_end_date IS NOT NULL AND t.actual_end_date > t.end_date THEN 1
                        WHEN t.status = 'completed' AND t.actual_end_date IS NULL AND t.updated_date::date > t.end_date THEN 1
                        WHEN t.status NOT IN ('completed', 'terminated', 'archived') AND t.end_date < '2026-07-14'::date THEN 1
                        ELSE 0 END)
                   OVER (PARTITION BY t.position_id
                         ORDER BY t.created_date ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS pos_past_overdue_rate
        FROM tasks_task t
        WHERE t.start_date IS NOT NULL AND t.end_date IS NOT NULL AND t.position_id IS NOT NULL
    ) sub
""")

base = base.merge(dept_agg[['id', 'dept_past_overdue_rate', 'dept_avg_revisions']], on='id', how='left')
base = base.merge(emp_agg[['id', 'emp_past_overdue_rate']], on='id', how='left')
base = base.merge(pos_agg[['id', 'pos_past_overdue_rate']], on='id', how='left')

global_mean_od = base['calculated_overdue'].mean()
base['dept_past_overdue_rate'] = base['dept_past_overdue_rate'].fillna(global_mean_od)
base['dept_avg_revisions'] = base['dept_avg_revisions'].fillna(0)
base['emp_past_overdue_rate'] = base['emp_past_overdue_rate'].fillna(global_mean_od)
base['pos_past_overdue_rate'] = base['pos_past_overdue_rate'].fillna(global_mean_od)

print(f'Dept rate: mean={base["dept_past_overdue_rate"].mean():.3f}, '
      f'null={(base["dept_past_overdue_rate"].isna().sum())}')
print(f'Emp rate: mean={base["emp_past_overdue_rate"].mean():.3f}')
print(f'Pos rate: mean={base["pos_past_overdue_rate"].mean():.3f}')

Dept rate: mean=0.588, null=0
Emp rate: mean=0.593
Pos rate: mean=0.573


---
## 11. Build Creation & Halfway Datasets

Separate the dual-column DataFrame into two clean CSVs.

In [16]:
wl_cols = [c for c in base.columns if c.startswith('wl_') and c != 'wl_low']

creation_col_map = {
    'status_encoded': 'status_encoded_creation',
    'approval_status_encoded': 'approval_status_encoded_creation',
    'lead_approval_status_encoded': 'lead_approval_status_encoded_creation',
    'ma_status_encoded': 'ma_status_encoded_creation',
    'ma_approval_status_encoded': 'ma_approval_status_encoded_creation',
    'num_ma_revisions': 'num_ma_revisions_at_creation',
    'kpi_is_overdue_flag': 'kpi_is_overdue_flag_creation',
    'kpi_status_ordinal': 'kpi_status_ordinal_creation',
    'num_kpi_revisions': 'num_kpi_revisions_at_creation',
    'num_revisions': 'num_revisions_at_creation',
    'revision_frequency': 'revision_frequency_at_creation',
    'revision_recency': 'revision_recency_at_creation',
    'num_subtasks': 'num_subtasks_at_creation',
    'has_subtasks': 'has_subtasks_at_creation',
    'subtask_completion_pct': 'subtask_completion_pct_at_creation',
    'subtask_overdue_rate': 'subtask_overdue_rate_at_creation',
    'task_comment_count': 'task_comment_count_at_creation',
    'avg_sub_status_changes': 'avg_sub_status_changes_at_creation',
}

halfway_col_map = {
    'status_encoded': 'status_encoded_halfway',
    'approval_status_encoded': 'approval_status_encoded_halfway',
    'lead_approval_status_encoded': 'lead_approval_status_encoded_halfway',
    'ma_status_encoded': 'ma_status_encoded_halfway',
    'ma_approval_status_encoded': 'ma_approval_status_encoded_halfway',
    'num_ma_revisions': 'num_ma_revisions_at_halfway',
    'kpi_is_overdue_flag': 'kpi_is_overdue_flag_halfway',
    'kpi_status_ordinal': 'kpi_status_ordinal_halfway',
    'num_kpi_revisions': 'num_kpi_revisions_at_halfway',
    'num_revisions': 'num_revisions_at_halfway',
    'revision_frequency': 'revision_frequency_at_halfway',
    'revision_recency': 'revision_recency_at_halfway',
    'num_subtasks': 'num_subtasks_at_halfway',
    'has_subtasks': 'has_subtasks_at_halfway',
    'subtask_completion_pct': 'subtask_completion_pct_at_halfway',
    'subtask_overdue_rate': 'subtask_overdue_rate_at_halfway',
    'subtask_completion_pct_at_halfway': 'subtask_completion_pct_at_halfway',
    'task_comment_count': 'task_comment_count_at_halfway',
    'avg_sub_status_changes': 'avg_sub_status_changes_at_halfway',
    'days_since_update': 'days_since_update_halfway',
}

static_cols = [
    'id', 'calculated_overdue', 'target_source',
    'planned_duration', 'creation_to_planned_start',
    'created_dow', 'created_is_weekend', 'created_is_friday',
    'created_month', 'created_quarter',
    'is_planned', 'risk_mapping', 'is_cross_dept', 'cross_dept_pair_exists',
    'dept_past_overdue_rate', 'dept_avg_revisions',
    'emp_past_overdue_rate', 'pos_past_overdue_rate',
] + wl_cols

In [17]:
ds_c = base[static_cols].copy()
for new_name, old_name in creation_col_map.items():
    if old_name in base.columns:
        ds_c[new_name] = base[old_name]

ds_h = base[static_cols].copy()
for new_name, old_name in halfway_col_map.items():
    if old_name in base.columns:
        ds_h[new_name] = base[old_name]

# Ensure no nulls and numeric types
for df in [ds_c, ds_h]:
    for col in df.columns:
        if df[col].dtype == 'object' and col not in ('id', 'target_source'):
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
        if col not in ('id', 'target_source'):
            df[col] = df[col].fillna(0)

creation_feats = [c for c in ds_c.columns if c not in ('id', 'calculated_overdue', 'target_source')]
halfway_feats = [c for c in ds_h.columns if c not in ('id', 'calculated_overdue', 'target_source')]

print(f'Creation dataset: {ds_c.shape[0]} rows, {ds_c.shape[1]} cols ({len(creation_feats)} features)')
print(f'Halfway dataset:  {ds_h.shape[0]} rows, {ds_h.shape[1]} cols ({len(halfway_feats)} features)')
print(f'Target rate: {ds_c["calculated_overdue"].mean():.2%}')

Creation dataset: 13895 rows, 36 cols (33 features)
Halfway dataset:  13895 rows, 38 cols (35 features)
Target rate: 47.81%


---
## 12. Leak Validation Checks

In [18]:
print('=== VALIDATION ===')
assert (ds_c['num_revisions'] == 0).all(), 'num_revisions at creation must be 0'
print('1. num_revisions at creation = 0 for ALL tasks: PASS')

assert (ds_h['num_revisions'] >= ds_c['num_revisions']).all(), 'revisions must not decrease'
print('2. num_revisions non-decreasing: PASS')

assert (ds_h['num_subtasks'] >= ds_c['num_subtasks']).all(), 'subtasks must not decrease'
print('3. num_subtasks non-decreasing: PASS')

zero_pct = (ds_c['status_encoded'] == 0).mean()
print(f'4. Status=not_started at creation: {zero_pct:.1%} (expect >=95%): {"PASS" if zero_pct >= 0.95 else "WARN"}')

assert ds_c['id'].equals(ds_h['id']), 'IDs must match'
print('5. Same IDs in both datasets: PASS')

assert ds_c['calculated_overdue'].equals(ds_h['calculated_overdue']), 'Target must match'
print('6. Same target in both datasets: PASS')

nulls_c = ds_c.isnull().sum().sum()
nulls_h = ds_h.isnull().sum().sum()
print(f'7. Nulls: creation={nulls_c}, halfway={nulls_h}: {"PASS" if nulls_c == 0 and nulls_h == 0 else "FAIL"}')

dropped = ['num_challenges', 'has_challenges', 'has_subtask_challenge', 'num_subtask_challenges',
           'has_kpi_challenge', 'num_kpi_challenges', 'has_kpi_potential_challenge',
           'num_kpi_potential_challenges', 'position_id_encoded']
for feat in dropped:
    assert feat not in ds_c.columns, f'{feat} should be dropped'
    assert feat not in ds_h.columns, f'{feat} should be dropped'
print(f'8. All {len(dropped)} dropped features absent: PASS')

print(f'\nAll checks passed.')

=== VALIDATION ===
1. num_revisions at creation = 0 for ALL tasks: PASS
2. num_revisions non-decreasing: PASS
3. num_subtasks non-decreasing: PASS
4. Status=not_started at creation: 1.0% (expect >=95%): WARN
5. Same IDs in both datasets: PASS
6. Same target in both datasets: PASS
7. Nulls: creation=0, halfway=0: PASS
8. All 9 dropped features absent: PASS

All checks passed.


---
## 13. Export to CSV

In [ ]:
output_dir = '../../data/clean'
os.makedirs(output_dir, exist_ok=True)
ds_c.to_csv(os.path.join(output_dir, 'dataset_at_creation_clean.csv'), index=False)
ds_h.to_csv(os.path.join(output_dir, 'dataset_at_halfway_clean.csv'), index=False)
print(f'Exported to {output_dir}/')

Exported to ../../data/v1/


In [20]:
print('\n=== CREATION FEATURES (33) ===')
creation_feature_list = [
    ('planned_duration', 'Derived', 'int', 'days from start to end'),
    ('creation_to_planned_start', 'Derived', 'int', 'days from created to start'),
    ('created_dow', 'Derived', 'int', 'day of week'),
    ('created_is_weekend', 'Derived', 'int', 'weekend flag'),
    ('created_is_friday', 'Derived', 'int', 'friday flag'),
    ('created_month', 'Derived', 'int', 'month'),
    ('created_quarter', 'Derived', 'int', 'quarter'),
    ('is_planned', 'Derived', 'int', 'planned flag'),
    ('risk_mapping', 'Derived', 'int', 'risk level'),
    ('is_cross_dept', 'Derived', 'int', 'cross-department assignment flag'),
    ('cross_dept_pair_exists', 'Derived', 'int', 'cross-dept pair exists'),
    ('status_encoded', 'DISTINCT ON', 'int', 'status from history at creation'),
    ('approval_status_encoded', 'DISTINCT ON', 'int', 'approval from history at creation'),
    ('lead_approval_status_encoded', 'DISTINCT ON', 'int', 'lead approval from history at creation'),
    ('ma_status_encoded', 'Static', 'int', 'MA status (unchanged)'),
    ('ma_approval_status_encoded', 'Static', 'int', 'MA approval status (unchanged)'),
    ('num_ma_revisions', 'FILTER(history)', 'int', 'MA revisions before creation'),
    ('kpi_is_overdue_flag', 'DISTINCT ON', 'int', 'KPI overdue from history at creation'),
    ('kpi_status_ordinal', 'DISTINCT ON', 'int', 'KPI status from history at creation'),
    ('num_kpi_revisions', 'FILTER(history)', 'int', 'KPI revisions before creation'),
    ('num_revisions', 'FILTER(history)', 'int', 'task revisions before creation (always 0)'),
    ('revision_frequency', 'FILTER(history)', 'float', '0 at creation'),
    ('revision_recency', 'FILTER(history)', 'int', 'days since creation'),
    ('num_subtasks', 'FILTER(created)', 'int', 'subtasks created before creation'),
    ('has_subtasks', 'FILTER(created)', 'int', 'has subtasks at creation'),
    ('subtask_completion_pct', 'FILTER(created)', 'float', 'subtask completion rate at creation'),
    ('subtask_overdue_rate', 'FILTER(created)', 'float', 'subtask overdue rate at creation'),
    ('task_comment_count', 'FILTER(created)', 'int', 'comments before creation'),
    ('avg_sub_status_changes', 'FILTER(history)', 'float', 'avg status changes per subtask'),
    ('dept_past_overdue_rate', 'Expanding Window', 'float', 'dept overdue rate before task'),
    ('dept_avg_revisions', 'Expanding Window', 'float', 'dept avg revisions before task'),
    ('emp_past_overdue_rate', 'Expanding Window', 'float', 'employee overdue rate before task'),
    ('pos_past_overdue_rate', 'Expanding Window', 'float', 'position overdue rate before task'),
]

print(f'{"Feature":35s} {"Type":20s} {"Dtype":8s} {"Description":s}')
print('-'*100)
for feat, ftype, dtype, desc in creation_feature_list:
    print(f'{feat:35s} {ftype:20s} {dtype:8s} {desc:s}')

print('\n=== HALFWAY EXTRA FEATURES (2 extra, total 35) ===')
halfway_extra = [
    ('subtask_completion_pct_at_halfway', 'FILTER(created)', 'float', 'subtask completion rate at halfway'),
    ('days_since_update', 'Derived', 'int', 'days since last update at halfway'),
]

print(f'{"Feature":35s} {"Type":20s} {"Dtype":8s} {"Description":s}')
print('-'*100)
for feat, ftype, dtype, desc in halfway_extra:
    print(f'{feat:35s} {ftype:20s} {dtype:8s} {desc:s}')

print('\n=== FEATURES WITH SAME NAME BUT DIFFERENT VALUES AT HALFWAY (14) ===')
overwritten = [
    ('status_encoded', 'DISTINCT ON', 'updated status from history at halfway'),
    ('approval_status_encoded', 'DISTINCT ON', 'updated approval at halfway'),
    ('lead_approval_status_encoded', 'DISTINCT ON', 'updated lead approval at halfway'),
    ('num_ma_revisions', 'FILTER(history)', 'more MA revisions by halfway'),
    ('kpi_is_overdue_flag', 'DISTINCT ON', 'updated KPI overdue at halfway'),
    ('kpi_status_ordinal', 'DISTINCT ON', 'updated KPI status at halfway'),
    ('num_kpi_revisions', 'FILTER(history)', 'more KPI revisions by halfway'),
    ('num_revisions', 'FILTER(history)', 'task revisions by halfway'),
    ('revision_frequency', 'FILTER(history)', 'updated at halfway'),
    ('revision_recency', 'FILTER(history)', 'days since last revision at halfway'),
    ('num_subtasks', 'FILTER(created)', 'more subtasks by halfway'),
    ('has_subtasks', 'FILTER(created)', 'updated flag at halfway'),
    ('task_comment_count', 'FILTER(created)', 'more comments by halfway'),
    ('avg_sub_status_changes', 'FILTER(history)', 'more churn by halfway'),
]

print(f'{"Feature":35s} {"Technique":20s} {"Note":s}')
print('-'*100)
for feat, tech, note in overwritten:
    print(f'{feat:35s} {tech:20s} {note:s}')



=== CREATION FEATURES (33) ===
Feature                             Type                 Dtype    Description
----------------------------------------------------------------------------------------------------
planned_duration                    Derived              int      days from start to end
creation_to_planned_start           Derived              int      days from created to start
created_dow                         Derived              int      day of week
created_is_weekend                  Derived              int      weekend flag
created_is_friday                   Derived              int      friday flag
created_month                       Derived              int      month
created_quarter                     Derived              int      quarter
is_planned                          Derived              int      planned flag
risk_mapping                        Derived              int      risk level
is_cross_dept                       Derived              int      cros

---
## Summary of Leakage Fixes Applied

| Technique | Features | Count |
|---|---|---|
| `FILTER (history_date <= cutoff)` | revisions, MA revisions, KPI revisions, sub-task churn | 4 feature groups |
| `created_date <= cutoff` | subtasks, comments | 2 feature groups |
| `DISTINCT ON` history lookup | status, approval_status, lead_approval_status, KPI is_overdue, KPI status | 5 features |
| Expanding window (self-excluding) | dept/emp/pos past_overdue_rate, dept_avg_revisions | 4 features |
| Dropped | 8 challenge features + position_id_encoded (fold-safe) | 9 features removed |

**Remaining: 4 group aggregate features** (`dept_past_overdue_rate`, `dept_avg_revisions`, `emp_past_overdue_rate`, `pos_past_overdue_rate`) still need fold-safe recomputation within CV folds in the training notebook.